In [7]:
import pandas as pd

In [11]:
account=pd.read_csv(r"account.csv", sep=";")
account.head()

,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102


In [12]:
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   account_id   4500 non-null   int64
 1   district_id  4500 non-null   int64
 2   frequency    4500 non-null   str  
 3   date         4500 non-null   int64
dtypes: int64(3), str(1)
memory usage: 140.8 KB


In [17]:
account.duplicated().sum()

np.int64(0)

In [18]:
account["account_id"].nunique()

4500

In [14]:
account["district_id"] = account["district_id"].astype(str)
account["account_id"] = account["account_id"].astype(str)
account["date"] = pd.to_datetime(account["date"].astype(str).str.zfill(6), format="%y%m%d")

In [16]:
freq_map = {"POPLATEK MESICNE": "monthly","POPLATEK TYDNE": "weekly","POPLATEK PO OBRATU": "after_transaction"}
account["frequency"] = account["frequency"].map(freq_map)

In [19]:
account["frequency"].value_counts()

frequency
monthly              4167
weekly                240
after_transaction      93
Name: count, dtype: int64

In [21]:
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   account_id   4500 non-null   str           
 1   district_id  4500 non-null   str           
 2   frequency    4500 non-null   str           
 3   date         4500 non-null   datetime64[us]
dtypes: datetime64[us](1), str(3)
memory usage: 140.8 KB


In [20]:
account.sample(5)

,account_id,district_id,frequency,date
3703,2381,1,monthly,1997-02-13
4446,3722,76,monthly,1997-12-07
4444,800,11,monthly,1997-12-06
3450,546,8,monthly,1996-11-23
2637,2474,33,monthly,1996-04-20


## Stage 5: Data Cleaning & Transformation — `account` Table

### 1. Data Cleaning & Transformation Log

| Column Name | Raw Type | Cleaned Type | Original / Raw Value | Converted / Mapped Value | Business / Risk Rationale |
| :--- | :---: | :---: | :--- | :--- | :--- |
| **`account_id`** | `int64` | `str` | Numeric integer (e.g., `1787`) | String object (e.g., `'1787'`) | Primary key. Converted to string to prevent numerical math operations and enable clean merging with `loan`, `disp`, and `trans` tables. |
| **`district_id`** | `int64` | `str` | Numeric integer (e.g., `18`) | String object (e.g., `'18'`) | Foreign key linking to the `district` (demographics) table. |
| **`date`** | `int64` | `datetime64[ns]` | YYMMDD integer string (e.g., `930705`) | ISO standard datetime (e.g., `1993-07-05`) | Represents account opening date. Essential for calculating "Account Tenure at Application" (Character C). Zero-padded via `.str.zfill(6)`. |
| **`frequency`** | `str` | `str` | Czech statement codes:<br>• `"POPLATEK MESICNE"`<br>• `"POPLATEK TYDNE"`<br>• `"POPLATEK PO OBRATU"` | English translation mappings:<br>• `'monthly'`<br>• `'weekly'`<br>• `'after_transaction'` | Standardized Czech terms into English for business readability and executive presentation. |

---

### 2. Validation & Integrity Audit Checklist

- [x] **Null Values:** 0 missing values detected across all columns.
- [x] **Primary Key Uniqueness:** `account_id` contains 4,500 unique values (100% unique, zero duplicates).
- [x] **Date Conversion:** Successfully converted to `YYYY-MM-DD` timestamps without parsing errors.
- [x] **Categorical Translation:** `frequency` successfully mapped with 0 unmapped (`NaN`) values remaining.